# 00_runtime_mode_and_user_config

In [ ]:
from __future__ import annotations
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path
RUN_PROFILE = os.environ.get('TSTW_RUN_PROFILE', 'real_video_vae_latent_probe_completion_formal').strip()
RUN_MODE = 'formal'
REQUIRE_FORMAL_PASS = True
DRIVE_ROOT = Path('/content/drive/MyDrive')
TSTW_ROOT = DRIVE_ROOT / 'TSTW'
MODELS_ROOT = DRIVE_ROOT / 'Models'
LOCAL_TSTW_ROOT = Path('/content/TSTW_runtime')
LOCAL_REPO_DIR = LOCAL_TSTW_ROOT / 'repo'
LOCAL_MODELS_DIR = LOCAL_TSTW_ROOT / 'models'
LOCAL_DATASET_CACHE_DIR = LOCAL_TSTW_ROOT / 'dataset_cache'
LOCAL_DATASET_DIR = LOCAL_TSTW_ROOT / 'datasets' / 'real_video_probe'
LOCAL_RUNS_DIR = LOCAL_TSTW_ROOT / 'runs'
LOCAL_TMP_DIR = LOCAL_TSTW_ROOT / 'tmp'
OVERRIDE_ROOT = TSTW_ROOT / 'configs' / RUN_PROFILE
DATASET_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'dataset_registry.json'
MODEL_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'model_registry.json'
DRIVE_STATE_PATH = TSTW_ROOT / 'registry' / 'drive_state.json'
RUNTIME_OVERRIDE_PATH = OVERRIDE_ROOT / 'runtime_override.json'
DATASET_OVERRIDE_PATH = OVERRIDE_ROOT / 'dataset_override.json'
MODEL_OVERRIDE_PATH = OVERRIDE_ROOT / 'model_override.json'
RESULTS_ROOT = TSTW_ROOT / 'results' / RUN_PROFILE
RESULT_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'result_registry.jsonl'
REPO_URL = os.environ.get('TSTW_VW_REPO_URL', 'https://github.com/your-org/TSTW-VW.git').strip()
REPO_BRANCH = os.environ.get('TSTW_VW_REPO_BRANCH', 'main').strip()
if not REPO_URL:
    raise ValueError('TSTW_VW_REPO_URL is required')
print({'run_profile': RUN_PROFILE, 'run_mode': RUN_MODE, 'repo_url': REPO_URL, 'repo_branch': REPO_BRANCH, 'drive_root': str(DRIVE_ROOT), 'local_repo_dir': str(LOCAL_REPO_DIR)})

# 01_mount_google_drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 02_read_drive_state_and_overrides

In [ ]:
for registry_path in (DATASET_REGISTRY_PATH, MODEL_REGISTRY_PATH, DRIVE_STATE_PATH):
    if not registry_path.exists():
        raise FileNotFoundError(registry_path)
for override_path in (RUNTIME_OVERRIDE_PATH, DATASET_OVERRIDE_PATH, MODEL_OVERRIDE_PATH):
    if not override_path.exists():
        raise FileNotFoundError(override_path)
dataset_registry = json.loads(DATASET_REGISTRY_PATH.read_text(encoding='utf-8'))
model_registry = json.loads(MODEL_REGISTRY_PATH.read_text(encoding='utf-8'))
drive_state = json.loads(DRIVE_STATE_PATH.read_text(encoding='utf-8'))
runtime_override = json.loads(RUNTIME_OVERRIDE_PATH.read_text(encoding='utf-8'))
dataset_override = json.loads(DATASET_OVERRIDE_PATH.read_text(encoding='utf-8'))
model_override = json.loads(MODEL_OVERRIDE_PATH.read_text(encoding='utf-8'))
dataset_key = str(dataset_override.get('dataset_key') or drive_state.get('default_dataset_key') or '').strip()
if not dataset_key:
    raise ValueError('dataset_key is required in dataset_override or drive_state')
runtime_override['run_profile'] = RUN_PROFILE
runtime_override['repo_url'] = REPO_URL
runtime_override['repo_branch'] = REPO_BRANCH
runtime_override['dataset_registry_path'] = str(DATASET_REGISTRY_PATH)
runtime_override['model_registry_path'] = str(MODEL_REGISTRY_PATH)
print({'dataset_registry': str(DATASET_REGISTRY_PATH), 'model_registry': str(MODEL_REGISTRY_PATH), 'drive_state': str(DRIVE_STATE_PATH), 'runtime_override': str(RUNTIME_OVERRIDE_PATH), 'dataset_override': str(DATASET_OVERRIDE_PATH), 'model_override': str(MODEL_OVERRIDE_PATH), 'dataset_key': dataset_key})

# 03_prepare_local_workspace

In [ ]:
LOCAL_TSTW_ROOT.mkdir(parents=True, exist_ok=True)
for runtime_path in (LOCAL_MODELS_DIR, LOCAL_DATASET_CACHE_DIR, LOCAL_DATASET_DIR, LOCAL_RUNS_DIR, LOCAL_TMP_DIR):
    if runtime_path.exists():
        shutil.rmtree(runtime_path)
    runtime_path.mkdir(parents=True, exist_ok=True)
if LOCAL_REPO_DIR.exists():
    shutil.rmtree(LOCAL_REPO_DIR)
LOCAL_REPO_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
if not RESULT_REGISTRY_PATH.parent.exists():
    RESULT_REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
print({'local_root': str(LOCAL_TSTW_ROOT), 'local_repo_dir': str(LOCAL_REPO_DIR), 'results_root': str(RESULTS_ROOT)})

# 04_clone_or_update_repository

In [ ]:
if (LOCAL_REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if any(LOCAL_REPO_DIR.iterdir()):
        shutil.rmtree(LOCAL_REPO_DIR)
        LOCAL_REPO_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(LOCAL_REPO_DIR)], check=True)
git_commit = subprocess.check_output(['git', '-C', str(LOCAL_REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
git_status_short = subprocess.check_output(['git', '-C', str(LOCAL_REPO_DIR), 'status', '--short'], text=True).strip()
short_commit = git_commit[:7]
print({'git_commit': git_commit, 'short_commit': short_commit, 'has_uncommitted_changes': bool(git_status_short)})

# 05_install_dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(LOCAL_REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', 'diffusers', 'accelerate', 'transformers', 'safetensors', 'opencv-python', 'imageio', 'imageio-ffmpeg', 'ffmpeg-python', 'numpy', 'pandas', 'scipy', 'scikit-image', 'tqdm', 'lpips', 'pytorch-msssim'], check=True)
pip_freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
print({'pip_freeze_lines': len([line for line in pip_freeze.splitlines() if line.strip()])})

# 06_copy_and_validate_dataset

In [ ]:
def _sha256_file(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with file_path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest().lower()


def _collect_dataset_tree_hints(root: Path, max_items: int = 30) -> dict:
    if not root.exists():
        return {'exists': False, 'sample_files': [], 'sample_dirs': []}
    sample_files = []
    for file_path in sorted(path for path in root.rglob('*') if path.is_file())[:max_items]:
        sample_files.append(str(file_path.relative_to(root)).replace('\\', '/'))
    sample_dirs = []
    for dir_path in sorted(path for path in root.rglob('*') if path.is_dir())[:max_items]:
        sample_dirs.append(str(dir_path.relative_to(root)).replace('\\', '/'))
    return {'exists': True, 'sample_files': sample_files, 'sample_dirs': sample_dirs}


def _resolve_cache_tar_path(dataset_override_payload: dict) -> Path:
    cache_tar_raw = str(dataset_override_payload.get('cache_tar_path', '')).strip()
    if not cache_tar_raw:
        raise RuntimeError(
            {
                'reason': 'dataset_cache_tar_path_missing',
                'expected_field': 'dataset_override.cache_tar_path',
                'recommended_example': '/content/drive/MyDrive/TSTW/datasets/cache/<dataset_key>.tar.zst',
            }
        )
    candidate = Path(cache_tar_raw)
    if candidate.is_absolute():
        return candidate

    # 支持相对路径写法：优先相对 TSTW_ROOT，其次相对 datasets/cache。
    relative_candidates = [
        TSTW_ROOT / cache_tar_raw,
        TSTW_ROOT / 'datasets' / 'cache' / cache_tar_raw,
    ]
    for resolved in relative_candidates:
        if resolved.exists():
            return resolved

    # 返回首选候选，后续统一抛可解释错误。
    return relative_candidates[0]


def _raise_dataset_error(reason: str, payload: dict) -> None:
    raise RuntimeError({'reason': reason, **payload})


def _validate_manifest_payload(manifest_payload: dict, dataset_root: Path) -> dict:
    if not isinstance(manifest_payload, dict):
        _raise_dataset_error('dataset_manifest_not_dict', {'dataset_root': str(dataset_root)})

    samples = manifest_payload.get('samples')
    if not isinstance(samples, list) or not samples:
        _raise_dataset_error(
            'dataset_manifest_samples_invalid',
            {
                'dataset_root': str(dataset_root),
                'expected': 'manifest.samples must be a non-empty list',
            },
        )

    missing_relpaths = []
    absolute_relpaths = []
    for sample in samples:
        relpath = str(sample.get('relpath', '')).strip()
        if not relpath:
            continue
        relpath_obj = Path(relpath)
        if relpath_obj.is_absolute():
            absolute_relpaths.append(relpath)
            continue
        resolved = dataset_root / relpath_obj
        if not resolved.exists():
            missing_relpaths.append(relpath)

    if absolute_relpaths:
        _raise_dataset_error(
            'dataset_manifest_relpath_must_be_relative',
            {
                'dataset_root': str(dataset_root),
                'absolute_relpaths': absolute_relpaths[:20],
            },
        )

    if missing_relpaths:
        _raise_dataset_error(
            'dataset_manifest_relpath_missing_files',
            {
                'dataset_root': str(dataset_root),
                'missing_relpaths': missing_relpaths[:50],
                'tree_hint': _collect_dataset_tree_hints(dataset_root),
            },
        )

    return {'sample_count': len(samples), 'missing_relpath_count': len(missing_relpaths)}


cache_tar_path = _resolve_cache_tar_path(dataset_override)
cache_tar_sha256 = str(dataset_override.get('cache_tar_sha256', '')).strip().lower()
if not cache_tar_path.exists():
    _raise_dataset_error(
        'dataset_cache_tar_not_found',
        {
            'cache_tar_path': str(cache_tar_path),
            'dataset_key': dataset_key,
            'recommended_location': str(TSTW_ROOT / 'datasets' / 'cache'),
        },
    )

if cache_tar_path.suffixes[-2:] != ['.tar', '.zst'] and cache_tar_path.suffix != '.zst':
    _raise_dataset_error(
        'dataset_cache_tar_suffix_invalid',
        {
            'cache_tar_path': str(cache_tar_path),
            'expected_suffix': '.tar.zst',
        },
    )

local_tar_path = LOCAL_DATASET_CACHE_DIR / cache_tar_path.name
shutil.copy2(cache_tar_path, local_tar_path)

actual_sha256 = _sha256_file(local_tar_path)
if cache_tar_sha256 and actual_sha256 != cache_tar_sha256:
    _raise_dataset_error(
        'dataset_cache_tar_sha256_mismatch',
        {
            'expected_sha256': cache_tar_sha256,
            'actual_sha256': actual_sha256,
            'local_tar_path': str(local_tar_path),
        },
    )

subprocess.run(['tar', '--zstd', '-xf', str(local_tar_path), '-C', str(LOCAL_DATASET_DIR)], check=True)

local_manifest_candidates = sorted(LOCAL_DATASET_DIR.rglob('dataset_manifest.json'))
if not local_manifest_candidates:
    _raise_dataset_error(
        'dataset_manifest_not_found_after_extract',
        {
            'local_dataset_root': str(LOCAL_DATASET_DIR),
            'tree_hint': _collect_dataset_tree_hints(LOCAL_DATASET_DIR),
        },
    )

local_dataset_manifest_path = local_manifest_candidates[0]
manifest_payload = json.loads(local_dataset_manifest_path.read_text(encoding='utf-8'))
manifest_stats = _validate_manifest_payload(manifest_payload, LOCAL_DATASET_DIR)

local_mp4_paths = list(LOCAL_DATASET_DIR.rglob('*.mp4'))
local_mp4_count = len(local_mp4_paths)
if local_mp4_count < 1:
    _raise_dataset_error(
        'dataset_mp4_not_found_after_extract',
        {
            'local_dataset_root': str(LOCAL_DATASET_DIR),
            'dataset_manifest_path': str(local_dataset_manifest_path),
            'tree_hint': _collect_dataset_tree_hints(LOCAL_DATASET_DIR),
        },
    )

runtime_override['local_dataset_root'] = str(LOCAL_DATASET_DIR)
runtime_override['dataset_manifest_path'] = str(local_dataset_manifest_path)
runtime_override['dataset_cache_tar_local_path'] = str(local_tar_path)
runtime_override['dataset_cache_tar_sha256_actual'] = actual_sha256
runtime_override['dataset_key'] = dataset_key

print({
    'dataset_tar_drive_path': str(cache_tar_path),
    'dataset_tar_local_path': str(local_tar_path),
    'dataset_tar_sha256_actual': actual_sha256,
    'dataset_manifest_path': str(local_dataset_manifest_path),
    'manifest_sample_count': manifest_stats['sample_count'],
    'mp4_count': local_mp4_count,
    'local_dataset_root': str(LOCAL_DATASET_DIR),
})

# 07_copy_and_validate_models

In [ ]:
def _as_path(value: object) -> Path:
    path_text = str(value or '').strip()
    if not path_text:
        return Path('')
    return Path(path_text)


def _compute_model_tree_digest(model_root: Path) -> str:
    entries = []
    for file_path in sorted(path for path in model_root.rglob('*') if path.is_file()):
        relpath = str(file_path.relative_to(model_root)).replace('\\', '/')
        size = int(file_path.stat().st_size)
        entries.append(f'{relpath}:{size}')
    payload = '\n'.join(entries).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def _collect_tree_hints(model_root: Path, max_files: int = 20) -> dict:
    if not model_root.exists():
        return {'exists': False, 'sample_files': [], 'sample_dirs': []}
    sample_files = []
    for file_path in sorted(path for path in model_root.rglob('*') if path.is_file())[:max_files]:
        sample_files.append(str(file_path.relative_to(model_root)).replace('\\', '/'))
    sample_dirs = []
    for dir_path in sorted(path for path in model_root.rglob('*') if path.is_dir())[:max_files]:
        sample_dirs.append(str(dir_path.relative_to(model_root)).replace('\\', '/'))
    return {'exists': True, 'sample_files': sample_files, 'sample_dirs': sample_dirs}


def _raise_vae_layout_error(local_vae_root: Path, hints: dict, candidates: list[Path]) -> None:
    candidate_text = [str(path.relative_to(local_vae_root)).replace('\\', '/') for path in candidates]
    raise RuntimeError(
        {
            'reason': 'vae_layout_invalid_for_formal',
            'local_vae_root': str(local_vae_root),
            'expected_layout': [
                'OptionA: <vae_root>/config.json + diffusion_pytorch_model.bin|safetensors',
                'OptionB: <vae_root>/<subdir>/config.json + diffusion_pytorch_model.bin|safetensors',
            ],
            'recommended_drive_model_override': {
                'vae.drive_model_root': '/content/drive/MyDrive/Models/vae',
                'note': 'Your cache may contain an extra level like vae/sd_vae_ft_mse/',
            },
            'candidate_model_roots': candidate_text,
            'tree_hint': hints,
        }
    )


def _resolve_model_roots(config: dict, model_name: str, required: bool) -> tuple[Path, Path]:
    default_drive_roots = {
        'vae': MODELS_ROOT / 'vae',
        'lpips': MODELS_ROOT / 'lpips',
        'clip': MODELS_ROOT / 'clip',
    }
    model_payload = config.get(model_name, {})
    drive_root = Path('')
    if isinstance(model_payload, dict):
        drive_root = _as_path(model_payload.get('drive_model_root'))
    if not str(drive_root):
        drive_root = _as_path(config.get(f'{model_name}_drive_model_root'))
    if not str(drive_root):
        drive_root = default_drive_roots[model_name]
    if not drive_root.exists():
        if required:
            raise FileNotFoundError(drive_root)
        return drive_root, Path('')
    local_root = LOCAL_MODELS_DIR / model_name
    return drive_root, local_root


def _copy_model_tree(drive_root: Path, local_root: Path, required: bool) -> str:
    if not drive_root.exists():
        if required:
            raise FileNotFoundError(drive_root)
        return ''
    if local_root.exists():
        shutil.rmtree(local_root)
    shutil.copytree(drive_root, local_root)
    file_count = len([path for path in local_root.rglob('*') if path.is_file()])
    if required and file_count < 1:
        raise RuntimeError({'empty_model_dir': str(local_root), 'drive_root': str(drive_root)})
    return str(local_root)


def _resolve_vae_load_root(local_vae_root: Path) -> Path:
    if not local_vae_root.exists():
        raise FileNotFoundError(local_vae_root)
    hints = _collect_tree_hints(local_vae_root)

    direct_config = local_vae_root / 'config.json'
    direct_weight_bin = local_vae_root / 'diffusion_pytorch_model.bin'
    direct_weight_st = local_vae_root / 'diffusion_pytorch_model.safetensors'
    direct_index = local_vae_root / 'model_index.json'
    if direct_config.exists() and (direct_weight_bin.exists() or direct_weight_st.exists() or direct_index.exists()):
        return local_vae_root

    candidates = []
    for config_path in local_vae_root.rglob('config.json'):
        candidate_root = config_path.parent
        has_weight = (
            (candidate_root / 'diffusion_pytorch_model.bin').exists()
            or (candidate_root / 'diffusion_pytorch_model.safetensors').exists()
            or (candidate_root / 'model_index.json').exists()
        )
        if has_weight:
            candidates.append(candidate_root)
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        candidates = sorted(candidates, key=lambda path: len(path.parts))
        return candidates[0]

    _raise_vae_layout_error(local_vae_root, hints, candidates)


def _resolve_lpips_weight(local_lpips_root: Path) -> Path:
    if not local_lpips_root.exists():
        raise FileNotFoundError(local_lpips_root)
    preferred = local_lpips_root / 'alex.pth'
    if preferred.exists():
        return preferred
    weight_files = sorted(local_lpips_root.rglob('*.pth'))
    if not weight_files:
        raise RuntimeError(
            {
                'reason': 'lpips_weight_not_found',
                'local_lpips_root': str(local_lpips_root),
                'expected_files': ['alex.pth', '*.pth'],
                'tree_hint': _collect_tree_hints(local_lpips_root),
            }
        )
    return weight_files[0]


vae_drive_root, vae_local_root = _resolve_model_roots(model_override, 'vae', required=True)
lpips_drive_root, lpips_local_root = _resolve_model_roots(model_override, 'lpips', required=True)
clip_drive_root, clip_local_root = _resolve_model_roots(model_override, 'clip', required=False)

local_vae_model_dir = _copy_model_tree(vae_drive_root, vae_local_root, required=True)
local_lpips_model_dir = _copy_model_tree(lpips_drive_root, lpips_local_root, required=True)
local_clip_model_dir = _copy_model_tree(clip_drive_root, clip_local_root, required=False)

resolved_vae_load_root = _resolve_vae_load_root(Path(local_vae_model_dir))
resolved_lpips_weight_path = _resolve_lpips_weight(Path(local_lpips_model_dir))

runtime_override['local_vae_model_root'] = str(resolved_vae_load_root)
runtime_override['local_lpips_model_root'] = str(Path(local_lpips_model_dir))
runtime_override['local_lpips_weight_path'] = str(resolved_lpips_weight_path)
runtime_override['local_clip_model_root'] = local_clip_model_dir or None
runtime_override['allow_mock_vae_backend'] = False
runtime_override['model_layout_digest'] = {
    'vae': _compute_model_tree_digest(Path(local_vae_model_dir)),
    'lpips': _compute_model_tree_digest(Path(local_lpips_model_dir)),
    'clip': _compute_model_tree_digest(Path(local_clip_model_dir)) if local_clip_model_dir else None,
}

print({
    'vae_drive_root': str(vae_drive_root),
    'lpips_drive_root': str(lpips_drive_root),
    'clip_drive_root': str(clip_drive_root),
    'local_vae_model_root': str(resolved_vae_load_root),
    'local_lpips_model_root': str(Path(local_lpips_model_dir)),
    'local_lpips_weight_path': str(resolved_lpips_weight_path),
    'local_clip_model_root': local_clip_model_dir or None,
    'allow_mock_vae_backend': runtime_override['allow_mock_vae_backend'],
})

# 08_check_gpu_and_runtime

In [ ]:
sys.path.insert(0, str(LOCAL_REPO_DIR))
from main.colab.runtime_check import run_runtime_preflight_check
runtime_check_report = run_runtime_preflight_check(run_mode=RUN_MODE, local_dataset_dir=LOCAL_DATASET_DIR, local_model_dirs=[Path(local_vae_model_dir), Path(local_lpips_model_dir)])
runtime_override['runtime_check_report'] = runtime_check_report
print(runtime_check_report)

# 09_verify_repository_contract

In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], cwd=LOCAL_REPO_DIR, check=True)
audit_output = subprocess.check_output([sys.executable, 'tools/harness/run_all_audits.py'], cwd=LOCAL_REPO_DIR, text=True)
print(audit_output[-1000:] if len(audit_output) > 1000 else audit_output)

# 10_run_unit_tests_smoke

In [ ]:
pytest_output = subprocess.check_output([sys.executable, '-m', 'pytest', '-q', '-m', 'smoke', 'tests/test_real_video_vae_latent_records_schema.py', 'tests/test_real_video_vae_latent_table_rebuild.py', 'tests/test_real_video_vae_latent_drive_packager.py', 'tests/test_real_video_vae_latent_quality_metrics.py', 'tests/test_real_video_vae_latent_temporal_metrics.py'], cwd=LOCAL_REPO_DIR, text=True)
print(pytest_output)

# 11_run_stage2_completion_formal

In [ ]:
construction_phase = 'real_video_vae_latent_probe'
utc_time = dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
dataset_short = ''.join(character for character in dataset_key if character.isalnum() or character in ['_', '-'])[:32] or 'dataset'
run_id = f"{construction_phase}__{RUN_PROFILE}__{dataset_short}__{utc_time}__{short_commit}"
run_root = LOCAL_RUNS_DIR / run_id
runtime_config_path = run_root / 'artifacts' / 'colab_real_video_vae_latent_runtime_config.json'
colab_runtime_manifest_path = run_root / 'artifacts' / 'colab_runtime_manifest.json'
run_root.mkdir(parents=True, exist_ok=True)
(run_root / 'artifacts').mkdir(parents=True, exist_ok=True)
runtime_override['run_id'] = run_id
runtime_override['git_commit'] = git_commit
runtime_override['dataset_manifest_snapshot_path'] = str(run_root / 'artifacts' / 'dataset_manifest_snapshot.json')
runtime_override['runtime_override_source'] = str(RUNTIME_OVERRIDE_PATH)
runtime_override['colab_runtime_manifest_overrides'] = {
    'repo_url': REPO_URL,
    'repo_branch': REPO_BRANCH,
    'git_status_short': git_status_short,
    'dataset_preflight': {
        'dataset_key': dataset_key,
        'dataset_tar_drive_path': str(cache_tar_path),
        'dataset_tar_local_path': str(local_tar_path),
        'dataset_tar_sha256_actual': actual_sha256,
        'dataset_manifest_path': str(local_dataset_manifest_path),
        'manifest_sample_count': manifest_stats['sample_count'],
        'mp4_count': local_mp4_count,
        'local_dataset_root': str(LOCAL_DATASET_DIR),
    },
    'model_preflight': {
        'vae_drive_root': str(vae_drive_root),
        'lpips_drive_root': str(lpips_drive_root),
        'clip_drive_root': str(clip_drive_root),
        'local_vae_model_root': str(resolved_vae_load_root),
        'local_lpips_model_root': str(Path(local_lpips_model_dir)),
        'local_lpips_weight_path': str(resolved_lpips_weight_path),
        'local_clip_model_root': local_clip_model_dir or None,
        'allow_mock_vae_backend': runtime_override['allow_mock_vae_backend'],
        'model_layout_digest': runtime_override['model_layout_digest'],
    },
    'runtime_preflight': runtime_check_report,
    'colab_runtime_manifest_path': str(colab_runtime_manifest_path),
}
runtime_config_path.write_text(json.dumps(runtime_override, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
sys.path.insert(0, str(LOCAL_REPO_DIR))
from main.protocol.real_video_vae_latent_runner import RealVideoVaeLatentRunner
runner = RealVideoVaeLatentRunner(LOCAL_REPO_DIR)
formal_result = runner.run(output_root=run_root, run_mode='formal', runtime_profile_override='formal', dataset_manifest_path=local_dataset_manifest_path, runtime_config_path=runtime_config_path)
print({'run_id': formal_result.run_id, 'run_root': str(run_root), 'event_records': len(formal_result.event_score_records), 'threshold_records': len(formal_result.threshold_records)})

# 12_rebuild_tables_and_reports

In [ ]:
from main.analysis.real_video_vae_latent_artifacts import RealVideoVaeLatentArtifactBuilder
rebuilt_paths = RealVideoVaeLatentArtifactBuilder().rebuild_artifacts(run_root)
dataset_manifest_src = LOCAL_REPO_DIR / 'configs' / 'data' / 'real_video_probe_manifest.json'
dataset_manifest_dst = run_root / 'artifacts' / 'dataset_manifest_snapshot.json'
shutil.copy2(dataset_manifest_src, dataset_manifest_dst)
(run_root / 'artifacts' / 'config_snapshot').mkdir(parents=True, exist_ok=True)
for config_path in (RUNTIME_OVERRIDE_PATH, DATASET_OVERRIDE_PATH, MODEL_OVERRIDE_PATH):
    shutil.copy2(config_path, run_root / 'artifacts' / 'config_snapshot' / config_path.name)
print({key: str(value) for key, value in rebuilt_paths.items()})

# 13_validate_formal_outputs

In [ ]:
from main.colab.notebook_result_checker import check_real_video_vae_latent_outputs
formal_checks = check_real_video_vae_latent_outputs(run_root, run_mode='formal', require_formal_pass_criteria=REQUIRE_FORMAL_PASS)
checks_path_local = run_root / 'artifacts' / 'checks.json'
checks_path_local.write_text(json.dumps(formal_checks, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
runtime_manifest_payload = json.loads(colab_runtime_manifest_path.read_text(encoding='utf-8'))
runtime_manifest_payload['formal_validation_summary'] = {
    'status': bool(formal_checks.get('status', False)),
    'decision': formal_checks.get('RealVideoVaeLatentDecision'),
    'blocking_reasons': formal_checks.get('BlockingReasons'),
    'next_allowed_stage': formal_checks.get('NextAllowedStage'),
    'require_formal_pass_criteria': REQUIRE_FORMAL_PASS,
    'checks_path': str(checks_path_local),
}
colab_runtime_manifest_path.write_text(json.dumps(runtime_manifest_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
if not formal_checks['status']:
    raise RuntimeError(formal_checks)
print(formal_checks)

# 14_pack_run_to_drive

In [ ]:
from main.colab.drive_packager import pack_real_video_vae_latent_run
from main.colab.tar_zst_packager import pack_run_to_tar_zst
compat_pack_root = run_root / 'artifacts' / 'compat_pack'
compat_pack_root.mkdir(parents=True, exist_ok=True)
compat_pack = pack_real_video_vae_latent_run(run_root=run_root, drive_output_dir=compat_pack_root, include_records=True, include_thresholds=True, include_tables=True, include_figures=True, include_reports=True, include_failure_gallery=True, include_manifest=True)
tar_pack = pack_run_to_tar_zst(run_root=run_root, drive_result_dir=RESULTS_ROOT, checks_payload=formal_checks, exclude_large_intermediate_latents=False)
drive_archive_path = tar_pack['archive_path']
drive_summary_path = tar_pack['summary_path']
drive_checks_path = tar_pack['checks_path']
print({'archive_path': str(drive_archive_path), 'summary_path': str(drive_summary_path), 'checks_path': str(drive_checks_path), 'compat_zip_path': str(compat_pack['zip_path'])})

# 15_update_result_registry

In [ ]:
registry_entry = {'schema_version': 'tstw_result_registry_entry.v1', 'run_id': run_id, 'run_profile': RUN_PROFILE, 'construction_phase': construction_phase, 'dataset_key': dataset_key, 'git_commit': git_commit, 'archive_path': str(drive_archive_path), 'summary_path': str(drive_summary_path), 'checks_path': str(drive_checks_path), 'decision': formal_checks.get('RealVideoVaeLatentDecision'), 'created_at': dt.datetime.utcnow().isoformat() + 'Z'}
with RESULT_REGISTRY_PATH.open('a', encoding='utf-8') as handle:
    handle.write(json.dumps(registry_entry, ensure_ascii=False) + '\n')
print(registry_entry)

# 16_print_final_summary

In [ ]:
log_dir = run_root / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
(log_dir / 'colab.log').write_text('Notebook completed successfully.\n', encoding='utf-8')
(log_dir / 'pytest.log').write_text(pytest_output, encoding='utf-8')
(log_dir / 'audit.log').write_text(audit_output, encoding='utf-8')
(log_dir / 'dependency_freeze.txt').write_text(pip_freeze, encoding='utf-8')
(log_dir / 'git_commit.txt').write_text(git_commit + '\n', encoding='utf-8')
(log_dir / 'git_status.txt').write_text(git_status_short + '\n', encoding='utf-8')
summary = {'run_id': run_id, 'run_root': str(run_root), 'decision': formal_checks.get('RealVideoVaeLatentDecision'), 'next_allowed_stage': formal_checks.get('NextAllowedStage'), 'drive_archive_path': str(drive_archive_path), 'drive_summary_path': str(drive_summary_path), 'drive_checks_path': str(drive_checks_path), 'result_registry_path': str(RESULT_REGISTRY_PATH)}
runtime_manifest_payload = json.loads(colab_runtime_manifest_path.read_text(encoding='utf-8'))
runtime_manifest_payload['drive_result_summary'] = {
    'archive_path': str(drive_archive_path),
    'summary_path': str(drive_summary_path),
    'checks_path': str(drive_checks_path),
    'result_registry_path': str(RESULT_REGISTRY_PATH),
}
runtime_manifest_payload['final_summary'] = summary
runtime_manifest_payload['result_registry_entry'] = registry_entry
colab_runtime_manifest_path.write_text(json.dumps(runtime_manifest_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(run_root / 'reports' / 'colab_final_summary.md').write_text('\n'.join([f'- {key}: {value}' for key, value in summary.items()]) + '\n', encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))